In [15]:
import os, requests
import numpy as np
import json
import pickle
import random
import re
import urllib.request
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score
from sklearn.datasets import load_files
import urllib.request, tarfile




np.random.seed(1337)
BASE_URL = os.getenv("BASE_URL", "http://154.57.164.71:31550")

def get_challenge(phase):
    r = requests.get(f"{BASE_URL}/challenge/{phase}")
    return r.json()

def submit_solutions(phase, solutions):
    r = requests.post(f"{BASE_URL}/submit/{phase}", json={"solutions": solutions})
    return r.json()

def predict(text):
    r = requests.post(f"{BASE_URL}/predict", json={"text": text})
    return r.json()




data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
dataset_path = data_dir / "aclImdb"

if not dataset_path.exists():
    print("[*] Downloading IMDB dataset...")
    url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
    tar_path = data_dir / "imdb.tar.gz"
    urllib.request.urlretrieve(url, tar_path)
    with tarfile.open(tar_path) as tar:
        tar.extractall(data_dir)
    tar_path.unlink()
    print("[+] Done")

# Load train split
train_data = load_files(str(dataset_path / "train"),
                        categories=["pos", "neg"],
                        encoding="utf-8", decode_error="replace")

df = pd.DataFrame({"message": train_data.data,
                   "label": ["pos" if t == 0 else "neg" for t in train_data.target]})

# load_files sorts categories alphabetically: neg=1, pos=0
# double-check:
print(train_data.target_names)  # should be ['neg', 'pos']
print(df['label'].value_counts())

['neg', 'pos']
label
neg    12500
pos    12500
Name: count, dtype: int64


In [2]:
import html as html_module
import unicodedata
def minimal_clean(text):
    """
    Minimal cleaning that preserves spam indicators.

    Parameters: text (str) raw SMS message
    Returns: str cleaned text with entities decoded, unicode normalized, and
             whitespace collapsed while keeping informative symbols.
    """
    # Decode HTML entities (e.g., &amp; -> &)
    text = html_module.unescape(text)

    # Normalize unicode characters
    text = unicodedata.normalize('NFKC', text)

    # Clean up excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\r+', ' ', text)

    return text.strip()


# WIN&nbsp;CASH&nbsp;NOW!!!\n\nClick&nbsp;here
# WIN CASH NOW!!! Click here

def clean_text(text):
    """
    Final cleaning for vectorization.

    Converts to lowercase and removes only problematic characters so that
    informative symbols remain available to the vectorizer.

    Parameters:
        text (str): Preprocessed message from `minimal_clean`.

    Returns:
        str: Normalized, whitespace‑collapsed text ready for tokenization.
    """
    text = text.lower()
    # Keep numbers, currency symbols, punctuation - they're spam features!
    # Only remove truly problematic characters
    text = re.sub(r'[^\w\s£$€¥!?.,;:\'\"-]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


print("[*] Applying minimal text cleaning (preserving spam indicators)...")
df['preprocessed'] = df['message'].apply(minimal_clean)

# Apply final cleaning for vectorization
df['clean_message'] = df['preprocessed'].apply(clean_text)

print("\n[*] Sample spam messages with preserved features:")
spam_samples = df[df['label'] == 'neg'].sample(3, random_state=42)
for idx, row in spam_samples.iterrows():
    msg = row['preprocessed'][:100] + "..." if len(row['preprocessed']) > 100 else row['preprocessed']
    print(f"  - {msg}")





# Remove only exact duplicates
original_size = len(df)
df = df.drop_duplicates(subset=['label', 'clean_message'])
print(f"\n[+] Removed {original_size - len(df)} duplicates")

# Remove empty messages
before_empty = len(df)
df = df[df['clean_message'].str.len() > 0]
print(f"[+] Removed {before_empty - len(df)} empty messages")




[*] Applying minimal text cleaning (preserving spam indicators)...

[*] Sample spam messages with preserved features:
  - I loved this movie. It is rare to get a glimpse of post-partum Vietnam, and this movie-sans combat s...
  - Although in my opinion this is one of the lesser musicals of stars Frank Sinatra, Gene Kelly, Kathry...
  - This animation has a very simple and straightforward good vs. evil plot and is all about action. Wha...

[+] Removed 97 duplicates
[+] Removed 0 empty messages


In [3]:
X = df['clean_message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n[+] Data split:")
print(f"    Training: {len(X_train)} messages")
print(f"    Testing: {len(X_test)} messages")

print("\n[*] Training Naive Bayes classifier...")

model_dir = Path("models")
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "sentiment_classifier.pkl"

if model_path.exists():
    print(f"[+] Loading saved model from {model_path}")
    with open(model_path, 'rb') as f:
        saved_data = pickle.load(f)
        vectorizer = saved_data['vectorizer']
        classifier = saved_data['classifier']

    # Transform data using existing vocabulary
    X_train_vec = vectorizer.transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

else:
    # Configure vectorizer to capture spam patterns
    vectorizer = CountVectorizer(
        max_features=3000,
        token_pattern=r'\b\w+\b|[£$€¥]+|\d+|!!+|\?\?+|\.\.+',
        lowercase=True,
        stop_words='english'
    )
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)


    classifier = MultinomialNB()
    classifier.fit(X_train_vec, y_train)

    # Save model for reproducibility
    with open(model_path, 'wb') as f:
        pickle.dump({'vectorizer': vectorizer, 'classifier': classifier}, f)
    print(f"[+] Model saved to {model_path}")



# Calculate accuracy scores
train_acc = classifier.score(X_train_vec, y_train)
test_acc = classifier.score(X_test_vec, y_test)
print(f"[+] Training accuracy: {train_acc:.4f}")
print(f"[+] Testing accuracy: {test_acc:.4f}")

# Get detailed predictions
y_pred = classifier.predict(X_test_vec)
print("\n[*] Classification Report:")
print(classification_report(y_test, y_pred))



[+] Data split:
    Training: 19922 messages
    Testing: 4981 messages

[*] Training Naive Bayes classifier...
[+] Loading saved model from models/sentiment_classifier.pkl
[+] Training accuracy: 0.8463
[+] Testing accuracy: 0.8388

[*] Classification Report:
              precision    recall  f1-score   support

         neg       0.84      0.83      0.84      2494
         pos       0.83      0.85      0.84      2487

    accuracy                           0.84      4981
   macro avg       0.84      0.84      0.84      4981
weighted avg       0.84      0.84      0.84      4981



In [10]:
print("\n[*] Simulating black-box attack scenario...")
print("[*] Budget: 1000 queries")

# Simulate limited query access
query_budget = 1000
queries_used = 0
query_log = []


# REPLACE extract_ham_word_freq with this
def extract_ham_word_freq(X_train, y_train, sample_size=500):
    pos_msgs = X_train[y_train == 'pos']
    neg_msgs = X_train[y_train == 'neg']
    
    limit = min(sample_size, len(pos_msgs))
    
    pos_freq = {}
    for msg in pos_msgs[:limit]:
        for w in str(msg).split():
            if 2 < len(w) < 15:
                pos_freq[w] = pos_freq.get(w, 0) + 1
    
    neg_freq = {}
    for msg in neg_msgs[:limit]:
        for w in str(msg).split():
            if 2 < len(w) < 15:
                neg_freq[w] = neg_freq.get(w, 0) + 1
    
    # Return ratio instead of raw freq — discriminative words only
    discriminative = {}
    for w, pc in pos_freq.items():
        nc = neg_freq.get(w, 0)
        if pc >= 5:
            discriminative[w] = pc / (nc + 1)  # high = appears way more in pos
    
    return discriminative


def estimate_budget_allocation(total_budget):
    """
    Estimate allocation across exploration, exploitation, and combination.

    Parameters
    ----------
    total_budget : int
        Total query budget available for discovery.

    Returns
    -------
    dict
        Mapping phase -> integer number of queries that sums to `total_budget`.
    """
    explore = int(0.4 * total_budget)
    exploit = int(0.4 * total_budget)
    combine = total_budget - explore - exploit  # absorb rounding
    return {
        'exploration': explore,
        'exploitation': exploit,
        'combination': combine,
    }





def select_high_frequency_words(word_freq, max_words=100, min_freq=5):
    """
    Select the most frequent ham words above a minimum frequency.

    Parameters
    ----------
    word_freq : dict[str, int]
        Token frequency table for sampled ham messages.
    max_words : int, default 100
        Maximum number of words to return.
    min_freq : int, default 5
        Minimum frequency a word must meet to be considered.

    Returns
    -------
    list[str]
        Top words sorted by decreasing frequency then lexicographically.
    """
    sorted_by_freq = sorted(word_freq.items(), key=lambda x: (-x[1], x[0]))
    top = [w for w, c in sorted_by_freq if c > min_freq][:max_words]
    return top



wf_example = extract_ham_word_freq(X_train, y_train, sample_size=500)

top_words_example = select_high_frequency_words(wf_example, max_words=100, min_freq=5)
print("[*] Example: select_high_frequency_words")
print(f"  Selected top words: {len(top_words_example)} (min_freq=5)")
print("  First 10:", ", ".join(top_words_example[:10]))


def merge_with_curated(top_words, additional_candidates=None):
    """
    Merge data-driven top words with curated conversational candidates.

    Parameters
    ----------
    top_words : list[str]
        High-frequency ham words from the previous step.
    additional_candidates : list[str] | None
        Optional curated list to include regardless of frequency.

    Returns
    -------
    list[str]
        Deduplicated merged list (lexicographically ordered).
    """
    if additional_candidates is None:
        additional_candidates = [
            "ok", "cos", "ill", "thats", "later", "said", "ask", "didnt",
            "dont", "doing", "going", "come", "home", "tomorrow", "today", "sorry",
            "thanks", "yeah", "yes", "sure", "see", "tell", "know", "think",
        ]
    merged = set(top_words) | set(additional_candidates)
    return sorted(merged)

merged_example = merge_with_curated(top_words_example)
added = sorted(set(merged_example) - set(top_words_example))
print("[*] Example: merge_with_curated")
print(f"  Merged size: {len(merged_example)} | Added curated: {len(added)}")
print("  Sample added terms:", ", ".join(added[:5]))



def build_candidate_vocabulary(
    X_train,
    y_train,
    sample_size=500,
    max_words=100,
    min_freq=5,
    additional_candidates=None,
):
    """
    Build a candidate vocabulary for black-box discovery from ham messages.

    Parameters
    ----------
    X_train : array-like of str
        Cleaned training messages.
    y_train : array-like of str
        Labels aligned with X_train ('ham' or 'spam').
    sample_size : int, default 500
        Number of ham messages to analyze.
    max_words : int, default 100
        Maximum number of top frequent ham words to keep before merging extras.
    min_freq : int, default 5
        Minimum frequency threshold for inclusion from the ham corpus.
    additional_candidates : list[str] | None
        Optional curated conversational terms to include.

    Returns
    -------
    list[str]
        Deduplicated candidate words ordered by decreasing ham frequency,
        then lexicographically for stable ties.
    """
    word_freq = extract_ham_word_freq(X_train, y_train, sample_size=sample_size)
    top_words = select_high_frequency_words(word_freq, max_words=max_words, min_freq=min_freq)
    merged = merge_with_curated(top_words, additional_candidates=additional_candidates)

    # Stable final ordering driven by ham frequency, then lexical for ties
    def sort_key(w):
        return (-word_freq.get(w, 0), w)

    return sorted(merged, key=sort_key)

cv_example = build_candidate_vocabulary(X_train, y_train)
print("[*] Example: build_candidate_vocabulary")
print(f"  Candidates: {len(cv_example)}")
print("  First 10:", ", ".join(cv_example[:10]))


# Build candidate vocabulary for discovery

candidate_words = [
    "excellent", "outstanding", "superb", "magnificent", "exceptional",
    "brilliant", "wonderful", "fantastic", "amazing", "incredible",
    "phenomenal", "spectacular", "marvelous", "splendid", "fabulous",
    "great", "awesome", "terrific", "impressive", "remarkable",
    "delightful", "lovely", "beautiful", "perfect", "stunning",
    "love", "loved", "enjoy", "enjoyed", "adore", "appreciate",
    "pleased", "happy", "glad", "satisfied", "delighted",
    "good", "nice", "pleasant", "enjoyable", "entertaining",
    "engaging", "captivating", "compelling", "interesting",
    "fun", "exciting", "thrilling", "inspiring", "uplifting",
    "recommend", "recommended", "masterpiece", "classic", "gem",
    "heartwarming", "best", "favorite", "positive", "charming"
]


print(f"[+] Testing {len(candidate_words)} candidate words extracted from ham messages")

print(f"\n[+] Total candidate words extracted: {len(candidate_words)}")
print("-" * 50)

# Print the words in rows of 10 for easy reading
for i in range(0, len(candidate_words), 10):
    row = candidate_words[i:i+10]
    print(", ".join(row))





[*] Simulating black-box attack scenario...
[*] Budget: 1000 queries
[*] Example: select_high_frequency_words
  Selected top words: 85 (min_freq=5)
  First 10: blob, worst, lame, superman, terrible,, redeeming, gadget, it?, puerto, waste
[*] Example: merge_with_curated
  Merged size: 109 | Added curated: 24
  Sample added terms: ask, come, cos, didnt, doing
[*] Example: build_candidate_vocabulary
  Candidates: 109
  First 10: blob, worst, lame, superman, terrible,, redeeming, gadget, it?, puerto, waste
[+] Testing 60 candidate words extracted from ham messages

[+] Total candidate words extracted: 60
--------------------------------------------------
excellent, outstanding, superb, magnificent, exceptional, brilliant, wonderful, fantastic, amazing, incredible
phenomenal, spectacular, marvelous, splendid, fabulous, great, awesome, terrific, impressive, remarkable
delightful, lovely, beautiful, perfect, stunning, love, loved, enjoy, enjoyed, adore
appreciate, pleased, happy, glad, satis

In [11]:


def estimate_budget_allocation(total_budget):
    """
    Estimate allocation across exploration, exploitation, and combination.

    Parameters
    ----------
    total_budget : int
        Total query budget available for discovery.

    Returns
    -------
    dict
        Mapping phase -> integer number of queries that sums to `total_budget`.
    """
    explore = int(0.4 * total_budget)
    exploit = int(0.4 * total_budget)
    combine = total_budget - explore - exploit  # absorb rounding
    return {
        'exploration': explore,
        'exploitation': exploit,
        'combination': combine,
    }

# Quick demo for budget allocation
allocation = estimate_budget_allocation(query_budget)
print("\n[*] Budget allocation:")
for phase, budget in allocation.items():
    print(f"  {phase:12}: {budget:4d} queries")
print(f"  Total: {sum(allocation.values())} / {query_budget}")




[*] Budget allocation:
  exploration :  400 queries
  exploitation:  400 queries
  combination :  200 queries
  Total: 1000 / 1000


In [12]:
def initialize_adaptive_scorer():
    """Initialize adaptive scoring data structures"""
    return {
        'word_scores': {},      # Maps word -> effectiveness score
        'word_counts': {},      # Maps word -> number of times tested
        'exploration_rate': 0.2  # 20% exploration for discovery phase
    }


def epsilon_greedy_select(scorer, available_words):
    """Select word using epsilon-greedy strategy

    Parameters:
        scorer (dict): Adaptive scorer state
        available_words (list): Candidate words to choose from

    Returns:
        str: Selected word for testing
    """
    import random

    if random.random() < scorer['exploration_rate']:
        # Exploration: try untested or rarely tested words
        untested = [w for w in available_words if w not in scorer['word_counts']]
        if untested:
            return random.choice(untested)
        else:
            # Choose least tested word
            return min(available_words,
                      key=lambda w: scorer['word_counts'].get(w, 0))



    else:
        # Exploitation: choose best performing word
        return max(available_words,
                  key=lambda w: scorer['word_scores'].get(w, 0))


def update_word_score(scorer, word, impact, alpha=0.3):
    """Update word score using exponential moving average

    Parameters:
        scorer (dict): Adaptive scorer state
        word (str): Word being scored
        impact (float): Observed reduction in spam probability
        alpha (float): Learning rate
    """
    if word not in scorer['word_scores']:
        scorer['word_scores'][word] = impact
        scorer['word_counts'][word] = 1
    else:
        # Exponential moving average
        old_score = scorer['word_scores'][word]
        scorer['word_scores'][word] = (1 - alpha) * old_score + alpha * impact
        scorer['word_counts'][word] += 1


def discover_word_combinations(message, test_words, max_size=3):
    """Discover effective word combinations through systematic search

    Parameters:
        message (str): Target spam message
        test_words (list): Promising words to test
        max_size (int): Maximum combination size

    Returns:
        dict: Mapping of word combinations to effectiveness scores
    """
    from itertools import combinations

    combination_scores = {}
    message_score = predict(message)['negative_probability']




    # Test individual words first
    for word in test_words[:20]:
        test_message = message + " " + word
        test_vec = vectorizer.transform([test_message])
        score = predict(test_message)['negative_probability']
        impact = message_score - score
        combination_scores[(word,)] = impact



    # Test pairs for synergy
    if max_size >= 2:
        for word1, word2 in combinations(test_words[:15], 2):
            test_message = message + " " + word1 + " " + word2
            test_vec = vectorizer.transform([test_message])
            score = predict(test_message)['negative_probability']

            # Calculate synergy
            individual_impact = combination_scores.get((word1,), 0) + combination_scores.get((word2,), 0)
            actual_impact = message_score - score
            synergy = actual_impact - individual_impact

            if synergy > 0:  # Positive synergy detected
                combination_scores[(word1, word2)] = actual_impact





    # Test triplets for top pairs
    if max_size >= 3:
        top_pairs = sorted(
            [(k, v) for k, v in combination_scores.items() if len(k) == 2],
            key=lambda x: x[1], reverse=True
        )[:5]

        for pair, pair_score in top_pairs:
            for word in test_words[:10]:
                if word not in pair:
                    triplet = tuple(sorted(pair + (word,)))
                    test_message = message + " " + " ".join(triplet)
                    test_vec = vectorizer.transform([test_message])
                    score = predict(test_message)['negative_probability']
                    combination_scores[triplet] = message_score - score

    return combination_scores



In [13]:
def three_phase_discovery(spam_messages, candidate_words, budget=1000):
    """Three-phase discovery: exploration, exploitation, combination

    Parameters:
        spam_messages (list): Target spam messages
        candidate_words (list): Vocabulary to test
        budget (int): Total query budget

    Returns:
        tuple: (discovered_words, combination_scores, queries_used)
    """
    scorer = initialize_adaptive_scorer()
    queries_used = 0

    # Allocate budgets using 40-40-20 split strategy
    allocation = estimate_budget_allocation(budget)
    exploration_budget = allocation['exploration']
    exploitation_budget = allocation['exploitation']
    combination_budget = allocation['combination']





    # Phase 1: Broad exploration (allocated budget)
    print(f"[*] Phase 1: Exploration (budget: {exploration_budget} queries)")

    p1_marks = {
        max(1, int(0.25 * exploration_budget)),
        max(1, int(0.50 * exploration_budget)),
        max(1, int(0.75 * exploration_budget)),
    }
    p1_reported = set()



    while queries_used < exploration_budget and len(candidate_words) > 0:
        test_message = random.choice(spam_messages)
        word = epsilon_greedy_select(scorer, candidate_words)
    
        # ✅ Use the actual challenge API, not the local model
        result_orig = predict(test_message)
        prob_orig = result_orig['negative_probability']
    
        result_aug = predict(test_message + " " + word)
        prob_aug = result_aug['negative_probability']
    
        impact = prob_orig - prob_aug
        update_word_score(scorer, word, impact)
        queries_used += 2
    
        # Milestone report — now correctly inside the loop
        if queries_used in p1_marks and queries_used not in p1_reported:
            top3 = sorted(scorer['word_scores'].items(), key=lambda x: x[1], reverse=True)[:3]
            print(
                f"  [P1 {queries_used}/{exploration_budget}] "
                f"tested_words={len(scorer['word_scores'])} | "
                f"top3=" + ", ".join(f"{w}:{s:.3f}" for w, s in top3)
            )
            p1_reported.add(queries_used)


    print(f"[+] Exploration complete. Queries: {queries_used}, Words tested: {len(scorer['word_scores'])}")
    top5 = sorted(scorer['word_scores'].items(), key=lambda x: x[1], reverse=True)[:5]
    if top5:
        print("  Top5 after exploration:")
        for w, s in top5:
            print(f"    {w:12} | score: {s:.3f}")



    # Phase 2: Focused exploitation
    scorer['exploration_rate'] = 0.1  # Reduce exploration

    # Get top words for exploitation
    top_words = sorted(scorer['word_scores'].items(), key=lambda x: x[1], reverse=True)[:30]
    top_word_list = [w for w, _ in top_words]

    print(f"\n[*] Phase 2: Exploitation (budget: {exploitation_budget} queries)")
    initial_queries = queries_used
    p2_mid = initial_queries + max(1, exploitation_budget // 2)



    while queries_used < initial_queries + exploitation_budget and len(top_word_list) > 0:
        test_message = random.choice(spam_messages[:20])  # Focus on fewer messages
        word = random.choice(top_word_list[:15])  # Focus on best words

        vec_orig = vectorizer.transform([test_message])
        prob_orig = predict(test_message)['negative_probability']

        vec_aug = vectorizer.transform([test_message + " " + word])
        prob_aug  = predict(test_message + " " + word)['negative_probability']

        impact = prob_orig - prob_aug
        update_word_score(scorer, word, impact)
        queries_used += 2

    print(f"[+] Exploitation complete. Total queries: {queries_used}")



    # Phase 3: Combination discovery (allocated budget)
    remaining_combo = combination_budget
    print(f"\n[*] Phase 3: Combination search (budget: {remaining_combo} queries)")

    best_combinations = {}
    combos_tested = 0



    if remaining_combo > 50:  # Need minimum queries for combinations
        for i in range(min(3, len(spam_messages))):
            if queries_used >= budget or remaining_combo <= 0:
                break

            test_msg = spam_messages[i]
            combos = discover_word_combinations(test_msg, top_word_list[:20], max_size=3)

            # Track best combinations across messages
            for combo, score in combos.items():
                if combo not in best_combinations or score > best_combinations[combo]:
                    best_combinations[combo] = score

            # Account for queries (~2 per combination) while respecting the budget
            to_add = min(remaining_combo, len(combos) * 2)
            queries_used += to_add
            remaining_combo -= to_add
            combos_tested += len(combos)

            # Midpoint snapshot
            if combination_budget > 0 and remaining_combo <= combination_budget // 2 and best_combinations:
                best = max(best_combinations.items(), key=lambda x: x[1])
                print(
                    f"  [P3 mid ~{combination_budget - remaining_combo}/{combination_budget}] "
                    f"combos_tested={combos_tested} | best={' + '.join(best[0])}:{best[1]:.3f}"
                )

            if remaining_combo <= 0:
                break


    print(f"[+] Combination search complete. Total queries: {queries_used}")

    # Return final results
    final_words = sorted(scorer['word_scores'].items(), key=lambda x: x[1], reverse=True)
    return final_words, best_combinations, queries_used


In [20]:
challenge = get_challenge('blackbox')
blackbox_reviews = [r['text'] for r in challenge['reviews']]
final_words, best_combos, queries = three_phase_discovery(
    blackbox_reviews, candidate_words, budget=1000
)

# Take top 5 words and repeat them to fill the full 40-word budget
# (same as strategy_2_repeated_words in the working script)
top_5 = [w for w, _ in final_words[:5]]
repeated = (top_5 * (40 // len(top_5) + 1))[:40]  # repeat to fill budget
top_words_to_inject = repeated

print(f"[*] Injecting repeated words: {top_5} x8 = {top_words_to_inject}")

solutions = []
for review in challenge['reviews']:
    augmented = review['text'] + " " + " ".join(top_words_to_inject)  # add all 40 at once
    
    result = predict(augmented)
    print(f"[{review['id']}] {result['label']} | pos={result['positive_probability']:.3f}")
    solutions.append({"id": review['id'], "augmented_text": augmented})

print(submit_solutions('blackbox', solutions))

[*] Phase 1: Exploration (budget: 400 queries)
  [P1 100/400] tested_words=10 | top3=splendid:0.003, uplifting:0.002, excellent:0.001
  [P1 200/400] tested_words=19 | top3=uplifting:0.001, excellent:0.001, terrific:0.001
  [P1 300/400] tested_words=27 | top3=uplifting:0.002, terrific:0.000, splendid:0.000
[+] Exploration complete. Queries: 400, Words tested: 34
  Top5 after exploration:
    brilliant    | score: 0.000
    terrific     | score: 0.000
    happy        | score: 0.000
    splendid     | score: 0.000
    uplifting    | score: 0.000

[*] Phase 2: Exploitation (budget: 400 queries)
[+] Exploitation complete. Total queries: 800

[*] Phase 3: Combination search (budget: 200 queries)
  [P3 mid ~200/200] combos_tested=156 | best=excellent + heartwarming + terrific:0.000
[+] Combination search complete. Total queries: 1000
[*] Injecting repeated words: ['brilliant', 'heartwarming', 'amazing', 'appreciate', 'incredible'] x8 = ['brilliant', 'heartwarming', 'amazing', 'appreciate', '

In [ ]:
print("as")